# Path B-lite: Similarity-weighted KDE

**Date:** 2026-04-18.
**Hypothesis:** The unscaled KDE's phase-1 under-prediction for high-volume movies is driven by equal weighting of training movies in KDE fitting. When `combined_score` selects 20 training movies, all 20 contribute equally to each critic's KDE shape regardless of how similar they are to the target.

**Intervention:** Weight each training movie's contribution by its combined_score similarity to target. High-similarity movies dominate KDE shape and base_rate estimation. Uses scipy's `gaussian_kde(weights=...)` — single-parameter extension, no new features.

**Same selection set as ship stack (top-20 combined_score), same bandwidth cap (0.7d). Only the internal weighting changes.**

**Predictions compared:**
- **Control:** ship stack — combined_score top-20 + equal-weighted KDE build + bandwidth cap 0.7d.
- **Candidate:** same selection → weighted KDE build (weights = combined_score).

**Metrics:** phase-1 MAE at T-3d on full cohort, stratified by actual_phase1 quartile, plus h/m subset breakdown.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    combined_score_selector, combined_score_with_scores,
    snapshot_state, passes_skip_rules_for_snap,
    build_critic_profiles, build_kde_lambda_model_capped,
    build_weighted_critic_profiles, build_weighted_kde_lambda_model,
    predict_window_custom,
    CACHE_DIR,
)

SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7
SNAP = 3.0

CACHE = CACHE_DIR / 'path_b_lite_weighted_kde.pkl'
print('Ready.')

## Quick sanity: weighted vs unweighted KDE shape on a single target

For `they_will_kill_you` (a known under-predicted high-vol target), compare phase 1 prediction with weighted vs unweighted KDE.

In [ ]:
sample = 'they_will_kill_you'
sg = gap_for_slug(sample)
sc = close_date_map[sample]
midnight_dbc = (sc - sc.floor('D')).total_seconds() / 86400
snap_time = sc - pd.Timedelta(days=SNAP)
state = snapshot_state(sample, snap_time)
tw = state['first_review_dbc'] - SNAP

scores = combined_score_with_scores(
    sample, sg, state['observed_critics'], tw,
    k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
)

# Unweighted
profiles_u = build_critic_profiles(reviews, close_date_map, list(scores.keys()), verbose=False)
model_u = build_kde_lambda_model_capped(profiles_u, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL)
pred_u = predict_window_custom(
    model_u, dbc_from=SNAP, dbc_to=midnight_dbc,
    observed_critics=state['observed_critics'],
    observed_count=state['observed_count'],
    first_review_dbc=state['first_review_dbc'],
)

# Weighted
profiles_w = build_weighted_critic_profiles(reviews, close_date_map, scores, verbose=False)
model_w = build_weighted_kde_lambda_model(profiles_w, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL)
pred_w = predict_window_custom(
    model_w, dbc_from=SNAP, dbc_to=midnight_dbc,
    observed_critics=state['observed_critics'],
    observed_count=state['observed_count'],
    first_review_dbc=state['first_review_dbc'],
)

# Actual
mr = reviews[reviews['movie_slug'] == sample].copy()
mr['dbc'] = (sc - mr['estimated_timestamp']).dt.total_seconds() / 86400
actual_p1 = int(((mr['dbc'] > midnight_dbc) & (mr['dbc'] <= SNAP)).sum())

print(f'{sample} at T-{SNAP:g}d:')
print(f'  actual_phase1:          {actual_p1}')
print(f'  unweighted KDE pred:    {pred_u:.2f}  (err {pred_u - actual_p1:+.2f})')
print(f'  weighted KDE pred:      {pred_w:.2f}  (err {pred_w - actual_p1:+.2f})')
print()
print(f'Training scores (top 5):')
for s, sc_v in sorted(scores.items(), key=lambda x: -x[1])[:5]:
    print(f'  {s:40s}  score={sc_v:.3f}')

## Full cohort LOO at T-3d

In [ ]:
def run_weighted_sweep(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    for i, target in enumerate(close_date_map):
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_window_days = state['first_review_dbc'] - SNAP
        if target_window_days <= 0:
            continue
        target_critics = state['observed_critics']

        scores = combined_score_with_scores(
            target, target_gap, target_critics, target_window_days,
            k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
        )
        if len(scores) < 5:
            continue

        # Ground truth
        mr = reviews[reviews['movie_slug'] == target].copy()
        mr['dbc'] = (target_close - mr['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((mr['dbc'] > midnight_utc_dbc) & (mr['dbc'] <= SNAP)).sum())

        # Control: unweighted (ship stack)
        profiles_u = build_critic_profiles(reviews, close_date_map, list(scores.keys()), verbose=False)
        model_u = build_kde_lambda_model_capped(
            profiles_u, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
        )
        pred_u = predict_window_custom(
            model_u, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
            observed_critics=target_critics,
            observed_count=state['observed_count'],
            first_review_dbc=state['first_review_dbc'],
        )

        # Candidate: weighted
        try:
            profiles_w = build_weighted_critic_profiles(
                reviews, close_date_map, scores, verbose=False,
            )
            model_w = build_weighted_kde_lambda_model(
                profiles_w, bandwidth_floor=SHIP_BANDWIDTH_FLOOR, bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
            )
            pred_w = predict_window_custom(
                model_w, dbc_from=SNAP, dbc_to=midnight_utc_dbc,
                observed_critics=target_critics,
                observed_count=state['observed_count'],
                first_review_dbc=state['first_review_dbc'],
            )
        except Exception as e:
            print(f'  Error for {target}: {e}')
            continue

        rows.append({
            'target': target,
            'target_gap': target_gap,
            'observed_count': state['observed_count'],
            'unweighted_pred': float(pred_u),
            'weighted_pred': float(pred_w),
            'actual_phase1': actual_p1,
            'err_unweighted': float(pred_u) - actual_p1,
            'err_weighted': float(pred_w) - actual_p1,
        })
        if (i + 1) % 30 == 0:
            print(f'  {i+1}/{len(close_date_map)} targets')

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

results = run_weighted_sweep()
print(f'\nn={len(results)}')

## Aggregate MAE by quartile

In [ ]:
results['abs_err_u'] = results['err_unweighted'].abs()
results['abs_err_w'] = results['err_weighted'].abs()
results['q_actual'] = pd.qcut(results['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

def summarize(df, label):
    mae_u = df['abs_err_u'].mean()
    mae_w = df['abs_err_w'].mean()
    me_u = df['err_unweighted'].mean()
    me_w = df['err_weighted'].mean()
    delta_pct = 100 * (mae_u - mae_w) / mae_u if mae_u > 0 else 0
    print(f'{label:42s}  n={len(df):3d}  ctrl_MAE={mae_u:6.2f}  weighted_MAE={mae_w:6.2f}  '
          f'delta={mae_u-mae_w:+6.2f}  {delta_pct:+6.1f}%  (me_u={me_u:+5.2f}, me_w={me_w:+5.2f})')

print('Ship stack (unweighted) vs weighted-KDE. Positive delta = weighted better.\n')
summarize(results, 'Full cohort')
print()
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    sub = results[results['q_actual'] == q]
    if len(sub):
        lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
        summarize(sub, f'{q} (actual_phase1 in [{lo}, {hi}])')

## H/m subset

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm = results[results['target'].isin(HM)].copy()
cols = ['target', 'actual_phase1', 'unweighted_pred', 'weighted_pred',
        'err_unweighted', 'err_weighted']
print('H/m subset:')
print(hm[cols].to_string(index=False, float_format='%.2f'))
print()
summarize(hm, 'H/m aggregate (5 movies)')

## Decision

Compare deltas across strata:

- **Full cohort non-worse AND high-vol (Q4 + h/m) improved** → weighted KDE is a real structural win, ship it.
- **Full cohort mildly regresses but high-vol improved a lot** → acceptable trade, but need to quantify magnitude.
- **Both strata move together (same direction)** → weighted KDE is essentially re-scaling predictions uniformly, not a shape fix.
- **No movement anywhere** → weighted KDE has no effect at n=20 training (normalization absorbs the signal).